# CDCo-MambaNet V4 — Google Colab Phase 2 Training Workflow

This notebook provides a complete, isolated training workflow for CDCo-MambaNet V4 on Google Colab (Tesla T4 GPU).

> **Strict Research Guardrail:** Do NOT run training until explicit authorization is granted.

## 1. Mount Google Drive & Clone Repository
Supports VM restarts without directory collision.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/CDCo_Mamba_Research_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

REPO_DIR = '/content/CDCO-Mamba-Research'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/poorna-19/Main-project-.git {REPO_DIR}
%cd {REPO_DIR}
!git checkout improved-fixes

## 2. Install Core Dependencies

In [ ]:
!pip install -q hydra-core>=1.3.2 omegaconf>=2.3.0 timm>=1.0.0 einops>=0.8.0 scikit-learn>=1.2.0 SimpleITK nilearn nibabel wandb

## 3. Verify Hardware & Compute Device

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**2):.1f} MB')

## 4. Run Safe Synthetic Verification (Zero Patient Data Accessed)
Verifies architecture parameters (3,226,317), autograd, gradient clipping and frozen split integrity in < 10 seconds.

In [ ]:
!python tools/safe_fresh_system_check.py

## 5. Prepare Dataset Paths & 4x sMRI Augmentation Cache
Ensure `abide_284.npy`, `abide_smri96.npy`, and `cohort.csv` are staged from Google Drive into `/content/data/abide/`.

In [ ]:
# Stage dataset files into fast local VM storage
!mkdir -p /content/data/abide
!cp /content/drive/MyDrive/abide_data/abide_284.npy /content/data/abide/
!cp /content/drive/MyDrive/abide_data/abide_smri96.npy /content/data/abide/
!cp /content/drive/MyDrive/abide_data/cohort.csv /content/data/abide/

# Pre-generate 4x sMRI augmentation disk cache if not already copied from Drive
!python tools/generate_smri_cache.py \
    --seed 0 \
    --dataset_path /content/data/abide/abide_284.npy \
    --out_cache /content/data/abide/train_aug_smri.npy \
    --split_path splits/exact_splits_seed0.json

## 6. Execute Phase 2 Experiment E1 (Pending Explicit Authorization)
Enforces exact E1 specification: 30 epochs, 171 warmup steps, 1710 total steps, 57 updates/epoch, accum 8, val_loss checkpoint selection.

In [ ]:
# DO NOT RUN until explicitly authorized
# %env WANDB_MODE=disabled
# !python __main__.py \
#     model=cdcomambanet_v4_unified \
#     dataset=ABIDE_lowmem \
#     optimizer=adamw_e1_cos \
#     training=e1_training \
#     dataset.path="/content/data/abide/abide_284.npy" \
#     dataset.aug_cache="/content/data/abide/train_aug_smri.npy" \
#     dataset.exact_splits_path="splits/exact_splits_seed0.json" \
#     repeat_time=1 seed=0 \
#     log_path="/content/drive/MyDrive/CDCo_Mamba_Research_Results/phase2_experiments/E1_lr_cosine"

## 7. Resume from Checkpoint (In Case of Colab Session Disconnect)
Restores full training state: model weights, optimizer state, scheduler step, epoch counter, and metric history.

In [ ]:
# In case of session interruption, uncomment and run:
# %env WANDB_MODE=disabled
# !python __main__.py \
#     model=cdcomambanet_v4_unified \
#     dataset=ABIDE_lowmem \
#     optimizer=adamw_e1_cos \
#     training=e1_training \
#     dataset.path="/content/data/abide/abide_284.npy" \
#     dataset.aug_cache="/content/data/abide/train_aug_smri.npy" \
#     dataset.exact_splits_path="splits/exact_splits_seed0.json" \
#     training.resume_checkpoint="/content/drive/MyDrive/CDCo_Mamba_Research_Results/phase2_experiments/E1_lr_cosine/seed_0/last_checkpoint.pt" \
#     repeat_time=1 seed=0 \
#     log_path="/content/drive/MyDrive/CDCo_Mamba_Research_Results/phase2_experiments/E1_lr_cosine"

## 8. Verify Checkpoint & Deliverable Persistence in Google Drive

In [ ]:
!ls -lh /content/drive/MyDrive/CDCo_Mamba_Research_Results/phase2_experiments/E1_lr_cosine/seed_0/